# NER Model: Performance Evaluation on CoNLL-2003
**Component:** Entity Extraction (Flair BiLSTM NER)  
**Model Used:** `ner-fast` (Flair, pretrained on CoNLL-2003)  
**Evaluation Dataset:** CoNLL-2003 Test Split (`eng.testb`)  

This notebook documents the baseline evaluation of the NER model used in EntiLytics, explains how the evaluation was conducted, and interprets the results in the context of published research benchmarks.

---
## Background

The CoNLL-2003 dataset is a standard benchmark in the Named Entity Recognition research community. It consists of English newswire text annotated with four entity types: Person (`PER`), Organization (`ORG`), Location (`LOC`), and Miscellaneous (`MISC`). Its variety of topics and entity types makes it well-suited for testing how accurately a system identifies and classifies named entities in news content.

Evaluation on this dataset follows a strict token-by-token matching approach. A predicted entity is considered correct only when every token in the entity span is labeled with the exact correct tag. Performance is measured using precision, recall, and F1-score, which together reflect both the accuracy and completeness of entity extraction.

Weber et al. (2023) note that while traditional F1-score metrics have known limitations, they do not distinguish between missing an entity entirely and classifying it under the wrong type, they remain the accepted standard for evaluating NER model performance across the research community.

---
## 1. Dataset Preparation

The CoNLL-2003 dataset is loaded from the Hugging Face datasets hub and converted into the BIO (Begin, Inside, Outside) text format required by the Flair library. Three splits are generated: training, validation, and test.

Only the **test split** (`test.txt`) is used for evaluation. The training and validation splits are loaded to satisfy the Flair `ColumnCorpus` loader but are not used during the evaluation run.

The BIO format represents each token on its own line with its entity label. A blank line separates sentences.

```
JAPAN  _  _  B-LOC
GET    _  _  O
LUCKY  _  _  O
WIN    _  _  O

Nadim  _  _  B-PER
Ladki  _  _  I-PER
```

In [ ]:
from datasets import load_dataset

# Load the CoNLL-2003 dataset from Hugging Face
dataset = load_dataset("lhoestq/conll2003")

# Integer IDs in the dataset map to these labels in order
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]


def save_as_conll(split_name: str, data) -> None:
    """
    Converts a Hugging Face dataset split into the CoNLL BIO text format
    required by the Flair ColumnCorpus loader.

    Format per line: Token _ _ Label
    Sentences are separated by blank lines.
    """
    output_file = f"{split_name}.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        for entry in data:
            for token, tag_id in zip(entry["tokens"], entry["ner_tags"]):
                f.write(f"{token} _ _ {label_list[tag_id]}\n")
            f.write("\n")
    print(f"Created: {output_file}")


save_as_conll("train", dataset["train"])
save_as_conll("valid", dataset["validation"])
save_as_conll("test", dataset["test"])

---
## 2. Baseline Model Evaluation

The pretrained `ner-fast` model is loaded from the Flair model hub and evaluated against the test split. `ner-fast` is a BiLSTM model with a Softmax output layer, meaning it makes independent per-token predictions without global sequence constraints.

The `evaluate()` function compares the model's predicted entity labels against the gold labels in `corpus.test` and computes precision, recall, and F1-score for each entity type and in aggregate.

In [ ]:
from pathlib import Path
from flair.data import Corpus
from flair.datasets import ColumnCorpus
from flair.models import SequenceTagger

base_path = Path.cwd()
columns = {0: "text", 3: "ner"}

# Load the three splits into a Corpus object.
# Only corpus.test is used during evaluation.
corpus: Corpus = ColumnCorpus(
    base_path, columns,
    train_file="train.txt",
    test_file="test.txt",
    dev_file="valid.txt"
)

print("Loading ner-fast model and evaluating on CoNLL-2003 test set...")
tagger = SequenceTagger.load("ner-fast")

result = tagger.evaluate(corpus.test, gold_label_type="ner")

print("\n" + "=" * 45)
print("CONLL-2003 BASELINE RESULTS")
print("=" * 45)
print(f"Overall F1-Score: {result.main_score:.4f}")
print("-" * 45)
print(result.detailed_results)
print("=" * 45)

---
## 3. Results

```
Overall F1-Score: 0.9147

              precision    recall  f1-score   support

         ORG     0.8676    0.9193    0.8927      1661
         LOC     0.9391    0.8873    0.9125      1668
         PER     0.9734    0.9734    0.9734      1617
        MISC     0.8513    0.8234    0.8371       702

   micro avg     0.9160    0.9134    0.9147      5648
   macro avg     0.9078    0.9008    0.9039      5648
weighted avg     0.9170    0.9134    0.9147      5648
```

The test set contains **5,648 total entity mentions** across all four types. For each mention, the model's predicted label sequence was compared against the gold label. Precision and recall were calculated from those comparisons, and micro-averaged F1 was computed as the harmonic mean of the two.

---
## 4. Interpretation of Results

### 4.1 Overall Performance

The model achieved a micro-averaged F1-score of **0.9147**, with precision of 0.9160 and recall of 0.9134. The two values being close to each other indicates the model is balanced.

### 4.2 Performance by Entity Type

| Entity Type | F1-Score | Observation |
|---|---|---|
| `PER` (Person) | 0.9734 | Best performing |
| `LOC` (Location) | 0.9125 | Strong |
| `ORG` (Organization) | 0.8927 | Good |
| `MISC` (Miscellaneous) | 0.8371 | Lowest |

The lower MISC score is consistent with findings across NER research. Because MISC groups unrelated entity types such as nationalities, events, and products into a single class.

### 4.3 Micro vs. Macro F1

The reported headline figure is **micro F1 (0.9147)**, which counts every individual entity token equally across all types. This is the standard reporting metric for CoNLL-2003 evaluation and is the number used for all benchmark comparisons.

The macro F1 (0.9039) averages across entity types equally, so the lower MISC score pulls the number down. This is useful for understanding per-class fairness but is not the primary evaluation metric for this benchmark.

---
## 5. Benchmark Comparison

To place the result in context, the achieved F1-score is compared against published BiLSTM-based NER research.

Guo et al. (2023) studied adversarial transfer learning for Named Entity Recognition across multiple domains including news and music. Their BiLSTM-Attention-CRF baseline model achieved the following domain-specific F1-scores:

- **News domain baseline: 88.46%** - the highest baseline across all tested domains, and the most directly comparable to EntiLytics since the system processes news articles
- **Music domain baseline: 74.29%** - the lowest baseline, included to show the full range

Their best method, the GRAD adversarial discriminator with private sharing mode, reached a peak F1-score of **92.99%** across all domains. This represents a specialized transfer learning model trained and fine-tuned specifically for its target domain.

| Reference | Model | Domain | F1-Score |
|---|---|---|---|
| Guo et al. (2023) | BiLSTM-Attention-CRF baseline | News | 88.46% |
| Guo et al. (2023) | BiLSTM-Attention-CRF baseline | Music | 74.29% |
| Guo et al. (2023) | GRAD (adversarial transfer learning) | Best domain | 92.99% |
| **EntiLytics** | **ner-fast (BiLSTM-Softmax, pretrained)** | **News (CoNLL-2003)** | **91.47%** |

The EntiLytics result of **91.47%** outperforms the news domain supervised baseline of 88.46% and falls within 1.5% of the peak GRAD result, which required a specialized adversarial training procedure with a purpose-built domain transfer framework.

This indicates that the pretrained `ner-fast` model, applied without any domain-specific fine-tuning, operates at a level consistent with strong research-grade NER performance for news content analysis.

---
## 6. Summary

| Metric | Value |
|---|---|
| Overall Micro F1 | 0.9147 |
| Precision | 0.9160 |
| Recall | 0.9134 |
| Best entity type | PER (0.9734) |
| Weakest entity type | MISC (0.8371) |
| Entities evaluated | 5,648 |

> The F1-score of 0.9147 establishes a strong baseline for the entity extraction component of EntiLytics. It outperforms the supervised BiLSTM baseline for the news domain reported by Guo et al. (2023) and approaches the performance of their specialized adversarial transfer learning model, using a pretrained model with no domain-specific fine-tuning.